
# Silver — PESQUISA DE URA

Atualiza a Silver de PESQUISA consumindo a Bronze (append-only) como **fonte de
streaming Delta** (`Trigger.AvailableNow`) e aplicando **MERGE** idempotente via
`foreachBatch`. O **checkpoint** controla o progresso; como `foreachBatch` tem
semântica **at-least-once**, a não-duplicidade vem do **MERGE por chave de negócio**.

**Fluxo**
1. Lê a Bronze como stream Delta, a partir do checkpoint.
2. Valida o **data contract**, faz parse do `body` (JSON) e deriva o período (helpers da lib).
3. Aplica um **gate de Data Quality** (`__dq_results`) antes do MERGE.
4. Aplica **MERGE** por `ID_CHAM`.



## Parâmetros


In [ ]:
# ===================== PARÂMETROS (Widgets) =====================
import sys

sys.path.append("/Workspace/Repos/data_master/Databricks/lib")

dbutils.widgets.text("catalog", "prd")
dbutils.widgets.text("bronze_schema", "b_dm_callcenter")
dbutils.widgets.text("silver_schema", "s_dm_callcenter")
dbutils.widgets.text("bronze_table", "surveys_once")
dbutils.widgets.text("silver_table", "tabe_pesq_ura")
dbutils.widgets.text("checkpoint_base", "/Volumes/prd/s_dm_callcenter/checkpoints/silver")

CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
BRONZE_TABLE  = dbutils.widgets.get("bronze_table")
SILVER_TABLE  = dbutils.widgets.get("silver_table")

BRONZE_FQN = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_FQN = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"
CHECKPOINT = f"{dbutils.widgets.get('checkpoint_base').rstrip('/')}/{SILVER_TABLE}"
DQ_RESULTS = f"{CATALOG}.{SILVER_SCHEMA}.__dq_results"
QUARANTINE = f"{CATALOG}.{SILVER_SCHEMA}.__quarantine"
METRICS    = f"{CATALOG}.{SILVER_SCHEMA}.__dataset_metrics"

from pyspark.sql import functions as F, types as T
from transforms import SilverStream, parse_body, rename_columns, add_load_audit
from quality import Expectation, run_observability

print("Bronze:", BRONZE_FQN)
print("Silver:", SILVER_FQN)
print("Checkpoint:", CHECKPOINT)


## Transformação


In [ ]:
# Data contract: schema esperado e campos obrigatórios (nomes crus da Bronze).
SCHEMA = T.StructType([
    T.StructField("id_chamada", T.StringType(),  True),
    T.StructField("id_pesquisa", T.StringType(),  True),
    T.StructField("data_envio", T.DateType(), True),
    T.StructField("nota", T.IntegerType(), True)
])
REQUIRED = ["id_chamada", "id_pesquisa", "nota"]


def transform(df_raw):
    rename = {
        'id_chamada':     'ID_CHAM',
        'id_pesquisa':    'ID_PESQ',
        'data_envio':     'DT_ENVI',
        'nota':           'VL_NOTA'
    }

    # parse + renomeação via helpers da lib
    df = parse_body(df_raw, SCHEMA)
    df = rename_columns(df, rename)

    # Período (a partir da data de envio) + auditoria de carga
    df = df.withColumn("CD_PERI", F.date_format(F.col("DT_ENVI"), "yyyyMM").cast("int"))
    df = add_load_audit(df)                            # DH_REFE_CRGA
    return df


## ▶️ Execução
Stream Delta da Bronze (`AvailableNow`) → `transform` → `MERGE` idempotente por `ID_CHAM`
via `foreachBatch`. O **checkpoint** controla o progresso.


In [ ]:
# Upsert incremental via streaming (AvailableNow) + foreachBatch(MERGE), com data
# contract (quarentena) e gate de DQ. O checkpoint controla o progresso; a
# idempotência (at-least-once sem duplicar) é garantida pelo MERGE por chave.
checks = [
    Expectation.not_null("ID_CHAM"),
    Expectation.unique("ID_CHAM"),
    Expectation.unique("ID_PESQ"),
    Expectation.between("VL_NOTA", 0, 10),
]

stream = SilverStream(spark)
stream.run(
    source_table_fqn=BRONZE_FQN,
    target_table_fqn=SILVER_FQN,
    transform=transform,
    keys=["ID_CHAM"],
    checkpoint_location=CHECKPOINT,
    cluster_by=["ID_PESQ"],
    expectations=checks,
    dq_results_table=DQ_RESULTS,
    contract_schema=SCHEMA,
    contract_required=REQUIRED,
    quarantine_table=QUARANTINE,
)
print(f"[OK] Silver atualizada → {SILVER_FQN}")

# Observabilidade de dataset: volume da data de envio mais recente vs média histórica.
obs = run_observability(
    spark,
    target_table_fqn=SILVER_FQN,
    metrics_table=METRICS,
    date_col="DT_ENVI",
)
print(obs.summary())
obs.raise_if_critical_failed()   # severidade padrão warn: sinaliza sem interromper